In [ ]:
# Example 1: Basic Serialization and Deserialization
import msgpack
import json
import sys

# Basic data structures for serialization
data = {
    'name': 'MessagePack Example',
    'version': 1.0,
    'features': ['fast', 'compact', 'cross-platform'],
    'config': {
        'enabled': True,
        'timeout': 30,
        'retries': 3
    },
    'metrics': [10.5, 20.3, 15.7, 8.9],
    'timestamp': 1638360000
}

print("Original data:")
print(f"Type: {type(data)}")
print(f"Content: {data}")
print("-" * 40)

# Serialize to MessagePack binary format
packed_data = msgpack.packb(data)
print(f"Packed data:")
print(f"Type: {type(packed_data)}")
print(f"Size: {len(packed_data)} bytes")
print(f"First 20 bytes: {packed_data[:20]}")
print("-" * 40)

# Deserialize back to Python objects
unpacked_data = msgpack.unpackb(packed_data)
print(f"Unpacked data:")
print(f"Type: {type(unpacked_data)}")
print(f"Content matches original: {unpacked_data == data}")
print(f"Content: {unpacked_data}")

# Compare with JSON for size efficiency
json_data = json.dumps(data).encode('utf-8')
print(f"\nSize comparison:")
print(f"MessagePack: {len(packed_data)} bytes")
print(f"JSON: {len(json_data)} bytes")
print(f"Space savings: {((len(json_data) - len(packed_data)) / len(json_data) * 100):.1f}%")
print("Basic serialization completed successfully!")

In [ ]:
# Example 2: Advanced Data Types and Custom Serialization
import msgpack
import datetime
import uuid
from decimal import Decimal

# Custom object for serialization
class CustomData:
    def __init__(self, name, value):
        self.name = name
        self.value = value
    
    def __repr__(self):
        return f"CustomData(name='{self.name}', value={self.value})"

# Advanced data types
advanced_data = {
    'binary_data': b'Hello MessagePack!',
    'large_integer': 2**63 - 1,
    'negative_integer': -2**31,
    'float_data': 3.141592653589793,
    'none_value': None,
    'empty_list': [],
    'empty_dict': {},
    'bytes_array': bytearray(b'mutable bytes'),
    'nested_structure': {
        'level1': {
            'level2': {
                'level3': ['deep', 'nesting', 'test']
            }
        }
    }
}

print("Advanced data types serialization:")
print(f"Original data keys: {list(advanced_data.keys())}")
print(f"Binary data: {advanced_data['binary_data']}")
print(f"Large integer: {advanced_data['large_integer']}")

# Serialize advanced types
packed_advanced = msgpack.packb(advanced_data)
unpacked_advanced = msgpack.unpackb(packed_advanced)

print(f"\nSerialization successful:")
print(f"Large integer preserved: {unpacked_advanced['large_integer'] == advanced_data['large_integer']}")
print(f"Binary data preserved: {unpacked_advanced['binary_data'] == advanced_data['binary_data']}")
print(f"Nested structure preserved: {unpacked_advanced['nested_structure'] == advanced_data['nested_structure']}")

# Custom serialization with hooks
def encode_custom(obj):
    """Custom encoder for special objects"""
    if isinstance(obj, datetime.datetime):
        return {'__datetime__': True, 'iso': obj.isoformat()}
    elif isinstance(obj, CustomData):
        return {'__custom__': True, 'name': obj.name, 'value': obj.value}
    elif isinstance(obj, Decimal):
        return {'__decimal__': True, 'value': str(obj)}
    raise TypeError(f"Object {obj} is not serializable")

def decode_custom(obj):
    """Custom decoder for special objects"""
    if '__datetime__' in obj:
        return datetime.datetime.fromisoformat(obj['iso'])
    elif '__custom__' in obj:
        return CustomData(obj['name'], obj['value'])
    elif '__decimal__' in obj:
        return Decimal(obj['value'])
    return obj

# Test custom serialization
special_data = {
    'timestamp': datetime.datetime.now(),
    'custom_obj': CustomData("test", 42),
    'precise_value': Decimal('3.141592653589793238462643383279')
}

print(f"\nCustom object serialization:")
print(f"Original timestamp: {special_data['timestamp']}")
print(f"Original custom object: {special_data['custom_obj']}")

packed_special = msgpack.packb(special_data, default=encode_custom)
unpacked_special = msgpack.unpackb(packed_special, object_hook=decode_custom)

print(f"Restored timestamp: {unpacked_special['timestamp']}")
print(f"Restored custom object: {unpacked_special['custom_obj']}")
print(f"Timestamp type preserved: {type(unpacked_special['timestamp'])}")
print("Advanced data types serialization completed!")

In [ ]:
# Example 3: Streaming and Large Data Handling
import msgpack
import io
import time

# Create large dataset for streaming
def generate_large_data(num_records=1000):
    """Generate large dataset for testing streaming"""
    for i in range(num_records):
        yield {
            'id': i,
            'name': f'record_{i:06d}',
            'data': [x * 0.1 for x in range(10)],
            'metadata': {
                'created': time.time() + i,
                'category': f'cat_{i % 5}',
                'active': i % 3 == 0
            }
        }

print("Streaming serialization with large datasets:")

# Method 1: Pack multiple objects into stream
stream_buffer = io.BytesIO()
packer = msgpack.Packer()

print("Packing 100 records into stream...")
start_time = time.time()
records_written = 0

for record in generate_large_data(100):
    packed = packer.pack(record)
    stream_buffer.write(packed)
    records_written += 1

pack_time = time.time() - start_time
stream_size = stream_buffer.tell()

print(f"Packed {records_written} records in {pack_time:.3f} seconds")
print(f"Stream size: {stream_size} bytes ({stream_size/1024:.1f} KB)")
print(f"Average record size: {stream_size/records_written:.1f} bytes")

# Method 2: Unpack from stream
print("\nUnpacking records from stream...")
stream_buffer.seek(0)
unpacker = msgpack.Unpacker(stream_buffer)

start_time = time.time()
records_read = 0
sample_records = []

for record in unpacker:
    records_read += 1
    if records_read <= 3:  # Keep first 3 for display
        sample_records.append(record)

unpack_time = time.time() - start_time

print(f"Unpacked {records_read} records in {unpack_time:.3f} seconds")
print(f"Records match: {records_read == records_written}")

print("\nSample unpacked records:")
for i, record in enumerate(sample_records):
    print(f"Record {i + 1}: ID={record['id']}, Name={record['name']}")
    print(f"  Metadata category: {record['metadata']['category']}")

# Method 3: Incremental packing with buffer management
print(f"\nIncremental packing with buffer management:")
buffer_size = 1024  # 1KB buffer
packer = msgpack.Packer()
total_chunks = 0
total_size = 0

for i, record in enumerate(generate_large_data(50)):
    packed = packer.pack(record)
    total_size += len(packed)
    
    # Simulate buffer management
    if (i + 1) % 10 == 0:  # Every 10 records
        total_chunks += 1
        print(f"Chunk {total_chunks}: Processed {i + 1} records, Size: {total_size} bytes")

print(f"Total chunks: {total_chunks}")
print(f"Total data size: {total_size} bytes")
print(f"Average chunk size: {total_size / total_chunks:.1f} bytes")
print("Streaming and large data handling completed!")

In [ ]:
# Example 4: Performance Benchmarking and Format Comparison
import msgpack
import json
import pickle
import time
import sys

# Generate test data of different sizes and complexities
def create_benchmark_data(size='medium'):
    """Create test data for performance benchmarking"""
    if size == 'small':
        return {'id': 1, 'name': 'test', 'values': [1, 2, 3]}
    elif size == 'medium':
        return {
            'metadata': {'version': '1.0', 'created': time.time()},
            'data': [{'id': i, 'value': i * 1.5, 'active': i % 2 == 0} for i in range(100)],
            'config': {'timeout': 30, 'retries': 3, 'debug': False}
        }
    else:  # large
        return {
            'records': [
                {
                    'id': f'record_{i:06d}',
                    'timestamp': time.time() + i,
                    'measurements': [x * 0.1 for x in range(20)],
                    'tags': [f'tag_{j}' for j in range(5)],
                    'metadata': {'category': f'cat_{i % 10}', 'priority': i % 3}
                }
                for i in range(500)
            ]
        }

# Benchmark function
def benchmark_format(data, format_name, serialize_func, deserialize_func, iterations=100):
    """Benchmark serialization format performance"""
    
    # Serialization benchmark
    start_time = time.time()
    serialized_data = None
    
    for _ in range(iterations):
        serialized_data = serialize_func(data)
    
    serialize_time = time.time() - start_time
    
    # Deserialization benchmark
    start_time = time.time()
    
    for _ in range(iterations):
        deserialized_data = deserialize_func(serialized_data)
    
    deserialize_time = time.time() - start_time
    
    return {
        'format': format_name,
        'serialize_time': serialize_time,
        'deserialize_time': deserialize_time,
        'total_time': serialize_time + deserialize_time,
        'size': len(serialized_data) if isinstance(serialized_data, bytes) else len(serialized_data.encode()),
        'data_matches': deserialized_data == data
    }

print("Performance benchmarking: MessagePack vs JSON vs Pickle")
print("=" * 60)

# Test with different data sizes
for data_size in ['small', 'medium', 'large']:
    test_data = create_benchmark_data(data_size)
    iterations = 1000 if data_size == 'small' else 100 if data_size == 'medium' else 10
    
    print(f"\nTesting with {data_size} dataset ({iterations} iterations):")
    print("-" * 40)
    
    # MessagePack benchmark
    msgpack_result = benchmark_format(
        test_data, 'MessagePack',
        msgpack.packb,
        msgpack.unpackb,
        iterations
    )
    
    # JSON benchmark  
    json_result = benchmark_format(
        test_data, 'JSON',
        lambda x: json.dumps(x).encode('utf-8'),
        lambda x: json.loads(x.decode('utf-8')),
        iterations
    )
    
    # Pickle benchmark
    pickle_result = benchmark_format(
        test_data, 'Pickle',
        pickle.dumps,
        pickle.loads,
        iterations
    )
    
    # Display results
    results = [msgpack_result, json_result, pickle_result]
    
    print(f"{'Format':<12} {'Serialize':<10} {'Deserialize':<12} {'Total':<8} {'Size':<8} {'Valid':<5}")
    print("-" * 65)
    
    for result in results:
        print(f"{result['format']:<12} "
              f"{result['serialize_time']:.4f}s    "
              f"{result['deserialize_time']:.4f}s      "
              f"{result['total_time']:.4f}s "
              f"{result['size']:<8} "
              f"{result['data_matches']}")
    
    # Speed comparison
    fastest_total = min(r['total_time'] for r in results)
    smallest_size = min(r['size'] for r in results)
    
    print(f"\nBest total time: {fastest_total:.4f}s")
    print(f"Smallest size: {smallest_size} bytes")
    
    msgpack_speedup = json_result['total_time'] / msgpack_result['total_time']
    msgpack_compression = (json_result['size'] - msgpack_result['size']) / json_result['size'] * 100
    
    print(f"MessagePack vs JSON:")
    print(f"  Speed improvement: {msgpack_speedup:.2f}x faster")
    print(f"  Size reduction: {msgpack_compression:.1f}% smaller")

print("\nPerformance benchmarking completed!")
print("MessagePack consistently shows better performance and compression!")

In [ ]:
# Example 5: Production Use Cases and Configuration Patterns
import msgpack
import io
import gzip
import hashlib
import time
from pathlib import Path

# Production configuration for different use cases
class MessagePackConfig:
    """Production configurations for different MessagePack use cases"""
    
    @staticmethod
    def web_api_config():
        """Optimized for web API responses"""
        return {
            'use_single_float': True,   # Reduce precision for bandwidth
            'use_bin_type': True,       # Use binary type for bytes
        }
    
    @staticmethod
    def high_performance_config():
        """Optimized for high-performance scenarios"""
        return {
            'use_single_float': False,  # Full precision
            'use_bin_type': True,
        }
    
    @staticmethod
    def storage_config():
        """Optimized for long-term storage"""
        return {
            'use_single_float': False,  # Full precision for storage
            'use_bin_type': True,
        }

# Production data cache implementation
class MessagePackCache:
    """Production-ready cache using MessagePack serialization"""
    
    def __init__(self, compress=True):
        self.cache = {}
        self.compress = compress
        self.stats = {'hits': 0, 'misses': 0, 'writes': 0}
    
    def _serialize(self, data):
        """Serialize with optional compression"""
        packed = msgpack.packb(data, **MessagePackConfig.storage_config())
        if self.compress:
            return gzip.compress(packed)
        return packed
    
    def _deserialize(self, data):
        """Deserialize with optional decompression"""
        if self.compress:
            data = gzip.decompress(data)
        return msgpack.unpackb(data)
    
    def put(self, key, value, ttl=None):
        """Store value in cache with optional TTL"""
        cache_entry = {
            'value': value,
            'timestamp': time.time(),
            'ttl': ttl
        }
        self.cache[key] = self._serialize(cache_entry)
        self.stats['writes'] += 1
    
    def get(self, key):
        """Retrieve value from cache"""
        if key not in self.cache:
            self.stats['misses'] += 1
            return None
        
        try:
            cache_entry = self._deserialize(self.cache[key])
            
            # Check TTL
            if cache_entry['ttl']:
                if time.time() - cache_entry['timestamp'] > cache_entry['ttl']:
                    del self.cache[key]
                    self.stats['misses'] += 1
                    return None
            
            self.stats['hits'] += 1
            return cache_entry['value']
        except Exception:
            self.stats['misses'] += 1
            return None
    
    def get_stats(self):
        """Get cache performance statistics"""
        total_requests = self.stats['hits'] + self.stats['misses']
        hit_rate = self.stats['hits'] / total_requests if total_requests > 0 else 0
        return {**self.stats, 'hit_rate': hit_rate}

print("Production use cases and configuration patterns:")
print("=" * 50)

# Use case 1: High-performance data serialization
print("1. High-Performance Data Processing:")
sample_data = [{'id': i, 'data': [x * 0.1 for x in range(50)]} for i in range(100)]

hp_config = MessagePackConfig.high_performance_config()
start_time = time.time()

# Serialize with high-performance config using packb (not Packer)
serialized_chunks = []

for item in sample_data:
    # Use packb which returns bytes directly
    chunk = msgpack.packb(item, **hp_config)
    serialized_chunks.append(chunk)

hp_time = time.time() - start_time
total_size = sum(len(chunk) for chunk in serialized_chunks)

print(f"  Processed {len(sample_data)} items in {hp_time:.4f}s")
print(f"  Total serialized size: {total_size} bytes")
print(f"  Average item size: {total_size / len(sample_data):.1f} bytes")

# Use case 2: Production cache implementation
print(f"\n2. Production Cache Implementation:")
cache = MessagePackCache(compress=True)

# Store different types of data
test_data = [
    ('user_123', {'name': 'Alice', 'email': 'alice@example.com', 'preferences': {'theme': 'dark'}}),
    ('session_abc', {'user_id': 123, 'expires': time.time() + 3600, 'permissions': ['read', 'write']}),
    ('config_app', {'version': '2.1.0', 'features': {'debug': False, 'analytics': True}}),
    ('metrics_daily', [{'timestamp': time.time() + i, 'value': i * 1.5} for i in range(20)])
]

for key, value in test_data:
    cache.put(key, value, ttl=300)  # 5-minute TTL

# Test cache retrieval
print("  Cache operations:")
for key, original_value in test_data[:2]:
    cached_value = cache.get(key)
    print(f"    {key}: {'HIT' if cached_value else 'MISS'} - Match: {cached_value == original_value}")

cache_stats = cache.get_stats()
print(f"  Cache stats: {cache_stats['hits']} hits, {cache_stats['misses']} misses")
print(f"  Hit rate: {cache_stats['hit_rate']:.2%}")

# Use case 3: Data integrity validation
print(f"\n3. Data Integrity and Validation:")

def create_data_package(data, metadata=None):
    """Create data package with integrity checking"""
    package = {
        'data': data,
        'metadata': metadata or {'created': time.time()},
        'version': '1.0'
    }
    
    # Serialize and create hash
    serialized = msgpack.packb(package, **MessagePackConfig.storage_config())
    data_hash = hashlib.sha256(serialized).hexdigest()
    
    return {
        'package': serialized,
        'hash': data_hash,
        'size': len(serialized)
    }

def verify_data_package(package_info):
    """Verify data package integrity"""
    computed_hash = hashlib.sha256(package_info['package']).hexdigest()
    return computed_hash == package_info['hash']

# Create and verify data packages
sample_package_data = {
    'transactions': [{'id': f'tx_{i}', 'amount': i * 10.5} for i in range(10)],
    'summary': {'total': 472.5, 'count': 10}
}

package_info = create_data_package(sample_package_data)
is_valid = verify_data_package(package_info)

print(f"  Data package created: {package_info['size']} bytes")
print(f"  Package hash: {package_info['hash'][:16]}...")
print(f"  Integrity check: {'PASSED' if is_valid else 'FAILED'}")

# Simulate corruption and re-verify
corrupted_package = dict(package_info)
corrupted_package['package'] = corrupted_package['package'][:-5] + b'ERROR'
is_corrupted_valid = verify_data_package(corrupted_package)
print(f"  Corrupted package check: {'PASSED' if is_corrupted_valid else 'FAILED (expected)'}")

# Use case 4: Compression comparison
print(f"\n4. Compression Efficiency:")
large_data = {
    'records': [
        {'id': i, 'name': f'record_{i}', 'values': [x * 0.5 for x in range(10)]}
        for i in range(100)
    ]
}

# Without compression
plain_packed = msgpack.packb(large_data)
# With compression
compressed_packed = gzip.compress(plain_packed)

compression_ratio = len(compressed_packed) / len(plain_packed)
space_saved = (1 - compression_ratio) * 100

print(f"  Original size: {len(plain_packed)} bytes")
print(f"  Compressed size: {len(compressed_packed)} bytes")
print(f"  Compression ratio: {compression_ratio:.2f}")
print(f"  Space saved: {space_saved:.1f}%")

print("\nProduction use cases demonstrated successfully!")
print("MessagePack provides excellent performance and flexibility for production systems!")